# Spark Exercises 09 — Capstone: Streamline Analytics

You are a data engineer at **Streamline**, an online store. Two worlds of data have landed: the **transactional** tables (orders, customers, products, regions) and the **behavioural** clickstream (`events.jsonl`). Your mission: build an end-to-end analytics pipeline that a business team could actually use — clean, enrich, rank, pivot, fuse transactions with behaviour into a *customer-360* table, and write the result. This capstone uses everything from chapters 1–8.

**Data:** `orders.csv`, `customers.csv`, `products.csv`, `regions.csv`, `events.jsonl`

### 1. **Setup** — open a local `SparkSession` (already written for you). In Foundry the session is created for you; locally we open one ourselves. `F` is the functions module — almost every column expression lives there.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (
    SparkSession.builder
    .appName("spark-course")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.ui.showConsoleProgress", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version)

### 2. Read all five sources: `orders`, `customers`, `products`, `regions` (CSV, inferSchema) and `events` (JSON).

### 3. **Clean the orders.** Keep only real sales (`status == 'completed'` and `quantity > 0`), and add `revenue = round(quantity * unit_price, 2)`. Call it `sales`. How many rows remain?

### 4. **Enrich.** Join `sales` to `customers`, then `regions` (on `region_id`), then `products` (on `product_sku`) into `fact`. Show `order_id, region_name, category, segment, revenue` for 5 rows.

### 5. **KPI 1 — revenue by region.** Total revenue per `region_name`, highest first.

### 6. **KPI 2 — a region × category matrix.** Use `pivot` to get total revenue with `region_name` as rows and `category` as columns.

### 7. **KPI 3 — top 3 products per region.** Sum revenue per (`region_name`, `product_name`), then use a window ranked by revenue within each region and keep the top 3.

### 8. **Customer value.** Per customer, compute `total_spent`, `n_orders`, and `last_order` (max `order_ts`). Call it `cust_value`. Show the top 5 spenders.

### 9. **Rank customers into spend deciles.** Add `decile = ntile(10)` over a window ordered by `total_spent` descending (decile 1 = top spenders). Show 10 rows.

### 10. **Behavioural side — the funnel.** From `events`, count events per `event_type`, ordered most to least (the classic page_view → search → add_to_cart → purchase funnel).

### 11. **Purchases by device.** Explode the `items` array of `purchase` events and compute purchase revenue (`qty * price`) per `device.os`.

### 12. **Fuse the two worlds — customer-360.** Build per-user event counts from `events` (rename `user_id` → `customer_id`), then **left join** `customers` → `cust_value` → event counts so every customer appears even with no orders/events. Fill missing numbers with 0. Show 10 rows of `customer_id, segment, total_spent, n_orders, n_events`.

### 13. **A KPI in pure SQL.** Register `c360` as a temp view and, with `spark.sql`, return average `total_spent` and average `n_events` **per `segment`**.

### 14. **Ship it.** Write the enriched `fact` table as Parquet, partitioned by `region_name`, into a temp folder. List the partition directories to confirm.